# DreamerV3 / R2-Dreamer: JAX vs PyTorch Timing Benchmark

**Goal:** Compare training throughput (steps/sec) and GPU memory usage across three model variants on identical hardware (H100).

**Variants:**
1. DreamerV3 (PyTorch, r2dreamer repo, rep_loss=dreamer) — includes decoder
2. R2-Dreamer (PyTorch, r2dreamer repo, rep_loss=r2dreamer) — no decoder
3. R2-Dreamer (JAX, our implementation) — no decoder

**Setup:** Synthetic data (64×64 RGB, 4 actions), batch=16, seq_len=64, size12M config. 30 timed training steps after warmup.

In [ ]:
import json, os, subprocess, sys, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RESULTS_PATH = os.path.join(REPO_ROOT, "output", "comparison", "habitat_timing.json")

# Run benchmark if results don't exist
if not os.path.exists(RESULTS_PATH):
    print("Running timing benchmark...")
    cmd = [sys.executable, os.path.join(REPO_ROOT, "scripts", "run_habitat_timing.py"),
           "--steps", "30", "--output", RESULTS_PATH]
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

with open(RESULTS_PATH) as f:
    results = json.load(f)
print(f"Loaded results from {RESULTS_PATH}")
for k, v in results.items():
    print(f"  {k}: {v.get('steps_per_sec', 'N/A'):.1f} steps/sec, {v.get('peak_gpu_gb', 'N/A'):.2f} GB")

In [ ]:
labels = []
sps_vals = []
mem_vals = []
label_map = {
    "dreamerv3_pytorch": "DreamerV3 (PyTorch)",
    "r2dreamer_pytorch": "R2-Dreamer (PyTorch)",
    "r2dreamer_jax": "R2-Dreamer (JAX)",
}
colors = {"dreamerv3_pytorch": "#4CAF50", "r2dreamer_pytorch": "#9C27B0", "r2dreamer_jax": "#F44336"}

for key in ["dreamerv3_pytorch", "r2dreamer_pytorch", "r2dreamer_jax"]:
    if key in results:
        labels.append(label_map[key])
        sps_vals.append(results[key]["steps_per_sec"])
        mem_vals.append(results[key]["peak_gpu_gb"])

# Reference for speedup
base_sps = sps_vals[0] if sps_vals else 1.0

print(f"{'Variant':<25} {'Steps/sec':>10} {'GPU (GB)':>10} {'Speedup':>10}")
print("-" * 60)
for l, s, m in zip(labels, sps_vals, mem_vals):
    print(f"{l:<25} {s:>10.1f} {m:>10.2f} {s/base_sps:>9.2f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = [colors[k] for k in ["dreamerv3_pytorch", "r2dreamer_pytorch", "r2dreamer_jax"] if k in results]

# Steps/sec
ax = axes[0]
bars = ax.bar(labels, sps_vals, color=bar_colors)
ax.set_ylabel("Steps/second")
ax.set_title("Training Throughput")
for bar, val in zip(bars, sps_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}", ha="center", va="bottom", fontsize=10)

# GPU Memory
ax = axes[1]
bars = ax.bar(labels, mem_vals, color=bar_colors)
ax.set_ylabel("Peak GPU Memory (GB)")
ax.set_title("GPU Memory Usage")
for bar, val in zip(bars, mem_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10)

plt.suptitle("Training Benchmark — size12M, B=16, T=64, 64×64 obs, H100", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, "output", "comparison", "timing_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## Conclusion

**Does JAX improve performance?**

[Results interpreted here after running]

Key observations:
- R2-Dreamer (no decoder) should be faster than DreamerV3 (with decoder) in both frameworks
- JAX vs PyTorch speed depends on JIT compilation efficiency and framework overhead
- GPU memory differences reflect decoder removal and framework memory management